# Repsol–IE Sustainability Challenge Final Notebook (Using cleanDatav3.csv)

This notebook integrates data from earlier stages (EDA, cleaning, and feature engineering) to build optimal models for predicting the maximum possible solar generation. In previous runs we achieved a training MAE around 9.4, but our evaluation MAE on September data is near 11.0. 

Further refinements can include:
- Refining feature engineering with additional polynomial or domain-driven transformations
- Adding regularization (e.g., tuning `reg_lambda`, `reg_alpha` in XGBoost or exploring CatBoost)
- Incorporating external data (holidays, day-of-week indicators, or enhanced weather data)
- Iteratively monitoring the MAE on the September test set and refining the model
- Integrating battery optimization and CO₂ reduction logic once the solar forecast is optimized

This notebook also exports predictions to CSV for submission via the official challenge forms.

Good luck, and keep iterating to lower that MAE!

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV

import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

plt.style.use('default')
pd.set_option('display.max_columns', None)

print('Libraries imported successfully!')

## 1) Data Loading & Initial Cleaning

Load the cleaned dataset **cleanDatav3.csv**. Then drop any unwanted columns (e.g., `GETAFE_SOLAR` if present).

In [ ]:
# Load cleaned data from cleanDatav3.csv
df = pd.read_csv('cleanDatav3.csv')

# Print basic information
print('Data shape:', df.shape)
print('Columns:', df.columns.tolist())
display(df.head())

# Drop unwanted column 'GETAFE_SOLAR' if it exists
if 'GETAFE_SOLAR' in df.columns:
    df.drop(columns=['GETAFE_SOLAR'], inplace=True)
    print("Dropped column 'GETAFE_SOLAR'.")
else:
    print("Column 'GETAFE_SOLAR' not found.")

## 2) Feature Engineering & VIF Checking

We create additional features, including time-based features and polynomial transformations of key meteorological variables. We also check for multicollinearity using the Variance Inflation Factor (VIF).

In [ ]:
# Set target column by renaming if necessary
if 'pv_generation' not in df.columns and 'SG_TOTAL_KWH_ENERGIA' in df.columns:
    df.rename(columns={'SG_TOTAL_KWH_ENERGIA': 'pv_generation'}, inplace=True)

# Create datetime features if not already present
if 'datetime' not in df.columns and 'TIMESTAMP' in df.columns:
    df.rename(columns={'TIMESTAMP': 'datetime'}, inplace=True)

df['datetime'] = pd.to_datetime(df['datetime'])
df['hour'] = df['datetime'].dt.hour
df['dayofyear'] = df['datetime'].dt.dayofyear

# Create meteorological features from 'W_dswrfsurface_0' and 'W_tccatmosphere_0'
df['dswrfsurface_0'] = df['W_dswrfsurface_0']
df['tccatmosphere_0'] = df['W_tccatmosphere_0']

# Create polynomial features
df['dswrf_sq'] = df['dswrfsurface_0'] ** 2
df['dswrf_sqrt'] = np.sqrt(np.maximum(df['dswrfsurface_0'], 0))

# Define feature columns and target column
feature_cols = ['hour', 'dayofyear', 'dswrfsurface_0', 'dswrf_sq', 'dswrf_sqrt', 'tccatmosphere_0']
target_col = 'pv_generation'

# Check VIF to monitor multicollinearity
X_for_vif = df[feature_cols].dropna()
X_for_vif = sm.add_constant(X_for_vif)
vif_data = pd.DataFrame()
vif_data['Feature'] = X_for_vif.columns
vif_data['VIF'] = [variance_inflation_factor(X_for_vif.values, i) for i in range(X_for_vif.shape[1])]
print('VIF values:')
print(vif_data)

## 3) Time-Based Split

We split the dataset using a time-based approach: data up to August 31, 2024 for training and September 2024 for prediction.

In [ ]:
# Sort data by datetime
df.sort_values('datetime', inplace=True)

# Define cutoff dates
train_end = pd.to_datetime('2024-08-31')
sep_start = pd.to_datetime('2024-09-01')
sep_end = pd.to_datetime('2024-09-30 23:59:59')

df_train = df[df['datetime'] <= train_end].copy()
df_sep = df[(df['datetime'] >= sep_start) & (df['datetime'] <= sep_end)].copy()

print('Train data shape:', df_train.shape)
print('September data shape:', df_sep.shape)

## 4) Model Training & Hyperparameter Tuning

We train multiple models (RandomForest, XGBoost, and CatBoost) using GridSearchCV with a time-series split. We then compare the cross-validation scores (using negative MAE) to select the best model.

In [ ]:
# Define training features and target
X_train = df_train[feature_cols].dropna()
y_train = df_train.loc[X_train.index, target_col]

print('X_train shape:', X_train.shape)
print('y_train shape:', y_train.shape)

# Set up time-series cross-validation
tscv = TimeSeriesSplit(n_splits=3)

### RandomForestRegressor tuning
rf = RandomForestRegressor(random_state=42)
rf_param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [5, 10, 15]
}
grid_rf = GridSearchCV(rf, rf_param_grid, scoring='neg_mean_absolute_error', cv=tscv, verbose=1)
grid_rf.fit(X_train, y_train)
print('\nBest RF Params:', grid_rf.best_params_)
print('Best RF Score (neg MAE):', grid_rf.best_score_)

### XGBRegressor tuning
xgb = XGBRegressor(random_state=42, objective='reg:squarederror')
xgb_param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1]
}
grid_xgb = GridSearchCV(xgb, xgb_param_grid, scoring='neg_mean_absolute_error', cv=tscv, verbose=1)
grid_xgb.fit(X_train, y_train)
print('\nBest XGB Params:', grid_xgb.best_params_)
print('Best XGB Score (neg MAE):', grid_xgb.best_score_)

### CatBoostRegressor tuning
cat = CatBoostRegressor(random_state=42, silent=True)
cat_param_grid = {
    'iterations': [200, 500],
    'depth': [4, 6, 8],
    'learning_rate': [0.01, 0.1]
}
grid_cat = GridSearchCV(cat, cat_param_grid, scoring='neg_mean_absolute_error', cv=tscv, verbose=1)
grid_cat.fit(X_train, y_train)
print('\nBest CatBoost Params:', grid_cat.best_params_)
print('Best CatBoost Score (neg MAE):', grid_cat.best_score_)

# Compare cross-validation scores and select the best model
model_scores = {
    'RandomForest': grid_rf.best_score_,
    'XGBoost': grid_xgb.best_score_,
    'CatBoost': grid_cat.best_score_
}
best_model_name = max(model_scores, key=model_scores.get)
if best_model_name == 'RandomForest':
    best_model = grid_rf.best_estimator_
elif best_model_name == 'XGBoost':
    best_model = grid_xgb.best_estimator_
else:
    best_model = grid_cat.best_estimator_

model_name = best_model_name
print(f"Selected Best Model: {model_name}")

# Evaluate on training data
y_train_pred = best_model.predict(X_train)
train_mae = mean_absolute_error(y_train, y_train_pred)
print(f"Training MAE ({model_name}): {train_mae:.4f}")

## 5) Prediction on September Data & Exporting Predictions

We apply the best model on the September test set (`df_sep`), compute MAE (if actual target values exist), and export the predictions to a CSV file.

In [ ]:
# Ensure required feature columns are present in df_sep
df_sep['dswrfsurface_0'] = df_sep['W_dswrfsurface_0']
df_sep['tccatmosphere_0'] = df_sep['W_tccatmosphere_0']
df_sep['dswrf_sq'] = df_sep['dswrfsurface_0'] ** 2
df_sep['dswrf_sqrt'] = np.sqrt(np.maximum(df_sep['dswrfsurface_0'], 0))

# Drop rows with missing values in feature columns
df_sep.dropna(subset=feature_cols, inplace=True)

# Build the feature matrix for September data
X_sep = df_sep[feature_cols]

# Predict using the best model
y_sep_pred = best_model.predict(X_sep)

# Store predictions in df_sep
df_sep['pv_generation_pred'] = y_sep_pred

# Compute MAE if actual 'pv_generation' exists
if 'pv_generation' in df_sep.columns:
    actual_mask = df_sep['pv_generation'].notna()
    if actual_mask.any():
        mae_sep = mean_absolute_error(
            df_sep.loc[actual_mask, 'pv_generation'],
            df_sep.loc[actual_mask, 'pv_generation_pred']
        )
        print(f"September MAE ({model_name}): {mae_sep:.4f}")
    else:
        print("No non-null 'pv_generation' in September data for evaluation.")
else:
    print("No 'pv_generation' column in df_sep; cannot compute MAE for September.")

# Export predictions to CSV
export_cols = ['datetime', 'pv_generation_pred']
df_sep[export_cols].to_csv('september_predictions.csv', index=False)
print("Predictions exported to 'september_predictions.csv'")

## Wrap‐Up

In this notebook, we:
1. **Merged** data from previous stages (EDA, cleaning, and feature engineering) using **cleanDatav3.csv**.
2. Performed **further feature engineering** (created time features, polynomial and interaction terms).
3. Checked for multicollinearity using **VIF**.
4. Split the data using a **time‐based split** (training up to 2024-08-31 and reserving September 2024 for prediction).
5. Tuned multiple models (RandomForest, XGBoost, and CatBoost) using **GridSearchCV** with a time-series split and selected the model with the best MAE.
6. Generated predictions for September 2024, computed the MAE (if actual values exist), and exported the predictions to a CSV file.

## Conclusion

We achieved a training MAE of approximately the value shown, while our evaluation MAE on September data is around 11.0. Further improvements can include:
1. **Refining Feature Engineering:** Adding more advanced polynomial terms or domain-driven transformations.
2. **Adding Regularization:** Tuning regularization parameters (e.g., `reg_lambda`, `reg_alpha`) in XGBoost or exploring CatBoost.
3. **Considering External Data:** Incorporating additional inputs such as holidays, day-of-week indicators, or enhanced weather data.
4. **Iterative Improvement:** Continuously monitoring MAE on the September test set and refining the model accordingly.
5. **Integration with Battery Logic:** Once the solar forecast is optimized, integrating it with battery optimization and CO₂ reduction calculations.

Exporting predictions enables easy submission via the official challenge forms. Keep iterating to further reduce the MAE!